# Makale Yeniden Üretimi: *Don't Trust Your Eyes: On the (Un)reliability of Feature Visualizations* (Geirhos ve ark., 2024)

**Makale:** R. Geirhos, R. S. Zimmermann, B. Bilodeau, W. Brendel, B. Kim. *Don't trust your eyes: on the (un)reliability of feature visualizations.* ICML 2024. [arXiv:2306.04719](https://arxiv.org/abs/2306.04719)

**Kitap bölümü:** Molnar, *Interpretable Machine Learning*, Bölüm 27 – Learned Features (Feature Visualization ve Network Dissection).

**Seçenek 1 – Zafiyet/Manipülasyon.** Makalenin iddiası: Feature visualization (bir birimin aktivasyonunu maksimize eden sentetik görüntü), ağın **doğal veri üzerindeki davranışından bağımsız olarak** manipüle edilebilir. Ağa gizli bir "devre" eklenirse, doğal görüntülerde ağ aynen eskisi gibi çalışırken feature visualization istenen (anlamsız ya da yanıltıcı) görüntüyü gösterir.

**Dürüstlük notu (önemli).** Makalenin devre tasarımını birebir kopyalamıyorum. Fikrini **sadeleştirilmiş bir yeniden uygulamayla** gösteriyorum: ağa, girdinin *yüksek frekans enerjisini* ölçen ve yalnızca doğal görüntülerde hiç görülmeyen düzeyin üstünde devreye giren bir kapı (gate) ekliyorum. Makaledeki asıl bulgu olan "FV, doğal davranışı yansıtmak zorunda değil" sonucunu kendi veri/modelimde sınıyorum.

**Bu defterde yapılanlar**

| Bölüm | Ne yapılıyor | Makaledeki karşılığı |
|---|---|---|
| 1 | Veri: Imagenette (ImageNet'in 10 sınıflık alt kümesi) | ImageNet |
| 2 | Ön eğitimli GoogLeNet (Inception V1), `inception4d` katmanı | Makale de Inception V1 kullanır |
| 3 | Aktivasyon maksimizasyonuyla feature visualization | Molnar §27.1, Olah ve ark. 2017 |
| 4 | **Gizli devre (fooling circuit) sadeleştirilmiş sürümü** | Makalenin ağ manipülasyonu |
| 5 | Niceleme: doğal davranış aynı mı, FV farklı mı? | Makalenin ana bulgusu |
| 6 | Görsel karşılaştırma: FV vs doğal en yüksek aktive eden görüntüler | Kitap: "eğitim örnekleri FV'yi tamamlar" |

In [ ]:
import copy, os, random, tarfile, urllib.request, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from PIL import Image

SEED = 0
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
COL_CLEAN, COL_FOOL = "#1f77b4", "#d62728"

# ---- Ayarlar ----
CAL_PER_CLASS = 20    # kalibrasyon kümesi: sınıf başına (toplam 200)
TEST_PER_CLASS = 10   # ayrı test kümesi: sınıf başına (toplam 100)
N_CH = 4              # incelenecek kanal sayısı
FV_STEPS = 256        # feature visualization adım sayısı
HF_MARGIN = 1.5       # kapı eşiği = doğal görüntülerdeki en yüksek HF enerjisi * HF_MARGIN
print("cihaz:", DEVICE, "| torch", torch.__version__)
if DEVICE == "cpu":
    print("UYARI: GPU yok. Çalışma zamanı türünü GPU yapın; yoksa FV optimizasyonu çok yavaş olur.")

## 1. Veri: Imagenette

[Imagenette](https://github.com/fastai/imagenette), ImageNet'in 10 kolay sınıflık alt kümesidir. Network Dissection'ın kullandığı Broden gibi piksel etiketli bir veri yerine burada, **feature visualization'ın doğal veriye karşı sınanması** için sıradan doğal görüntüler yeterlidir. İki ayrı küme kullanıyorum:

* **Kalibrasyon kümesi:** kapı eşiğini ve genliği belirlemek, en yüksek aktive eden doğal görüntüleri bulmak için,
* **Ayrı test kümesi:** doğal davranışın değişmediğini *görmediği* görüntülerde sınamak için.

In [ ]:
URL = "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz"
ARCHIVE, ROOT = "imagenette2-320.tgz", "imagenette2-320"

if not os.path.exists(ROOT):
    print("Imagenette indiriliyor (~325 MB)...")
    urllib.request.urlretrieve(URL, ARCHIVE)
    with tarfile.open(ARCHIVE) as t:
        try:
            t.extractall(".", filter="data")
        except TypeError:
            t.extractall(".")

# Imagenette sınıfı -> ImageNet-1k sınıf indeksi (ön eğitimli modelin çıktısıyla eşleşir)
SINIF = {
    "n01440764": ("tench", 0),
    "n02102040": ("English springer", 217),
    "n02979186": ("cassette player", 482),
    "n03000684": ("chain saw", 491),
    "n03028079": ("church", 497),
    "n03394916": ("French horn", 566),
    "n03417042": ("garbage truck", 569),
    "n03425413": ("gas pump", 571),
    "n03445777": ("golf ball", 574),
    "n03888257": ("parachute", 701),
}

rows = []
for wnid in sorted(os.listdir(f"{ROOT}/val")):
    if wnid not in SINIF:
        continue
    d = f"{ROOT}/val/{wnid}"
    for fn in sorted(os.listdir(d)):
        if fn.lower().endswith((".jpeg", ".jpg", ".png")):
            rows.append(dict(path=f"{d}/{fn}", wnid=wnid, sinif=SINIF[wnid][0],
                             imagenet_idx=SINIF[wnid][1]))
veri = pd.DataFrame(rows)
veri.to_csv("veri.csv", index=False)      # veri manifesti: repoya bunu da koyun
print(veri.shape)
print(veri.sinif.value_counts().to_string())

prep = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor()])

def load_img(path):
    return prep(Image.open(path).convert("RGB"))     # (3,224,224), [0,1]

def load_batch(paths):
    return torch.stack([load_img(p) for p in paths])

class Normalized(nn.Module):
    """[0,1] aralığındaki görüntüyü alır, ImageNet normalizasyonunu model içinde uygular.
    Böylece saldırı ve saliency hesapları piksel uzayında ([0,1]) yapılır."""
    def __init__(self, m):
        super().__init__()
        self.m = m
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))
    def forward(self, x):
        return self.m((x - self.mean) / self.std)

In [ ]:
cal = veri.groupby("wnid").sample(n=CAL_PER_CLASS, random_state=SEED)
rest = veri.drop(cal.index)
test = rest.groupby("wnid").sample(n=TEST_PER_CLASS, random_state=SEED)
X_cal, X_test = load_batch(cal.path), load_batch(test.path)
print("kalibrasyon:", tuple(X_cal.shape), "| test:", tuple(X_test.shape))

## 2. Model ve "birim" tanımı

Model: ImageNet'te ön eğitimli **GoogLeNet (Inception V1)**. Birim olarak `inception4d` çıktısındaki bir **kanal**ı alıyorum (kitaptaki "convolution channel" birimi); kanalın skoru = o kanalın uzamsal ortalama aktivasyonu:

$$h_z(\mathbf{x}) = \frac{1}{|U||V|}\sum_{u,v} h_{n,u,v,z}(\mathbf{x})$$

`UnitProbe` sınıfı katmana bir *forward hook* koyar; böylece (a) aktivasyonu okuyabiliriz, (b) ileride aynı noktaya gizli devreyi ekleyebiliriz.

In [ ]:
base = torchvision.models.googlenet(weights=torchvision.models.GoogLeNet_Weights.IMAGENET1K_V1)

class UnitProbe:
    """GoogLeNet 'inception4d' çıktısına hook koyar. gate verilirse seçili kanala gizli bir terim ekler."""
    def __init__(self, backbone):
        self.model = Normalized(backbone).eval().to(DEVICE)
        for p in self.model.parameters():
            p.requires_grad_(False)
        self.channel, self.amp, self.gate = None, 0.0, None
        self._x, self.act = None, None
        backbone.inception4d.register_forward_hook(self._hook)

    def _hook(self, module, inp, out):
        if self.gate is not None and self.channel is not None:
            g = self.gate(self._x)                                   # (B,)
            out = out.clone()
            out[:, self.channel] = out[:, self.channel] + self.amp * g.view(-1, 1, 1)
        self.act = out
        return out

    def forward_all(self, x):
        self._x = x                                                  # [0,1] aralığında ham görüntü
        logits = self.model(x)
        return logits, self.act

probe_clean = UnitProbe(copy.deepcopy(base))

@torch.no_grad()
def unit_scores(probe, X, bs=50):
    S, L = [], []
    for i in range(0, len(X), bs):
        lg, act = probe.forward_all(X[i:i + bs].to(DEVICE))
        S.append(act.mean((2, 3)).cpu()); L.append(lg.cpu())
    return torch.cat(S), torch.cat(L)

S_cal, _ = unit_scores(probe_clean, X_cal)
print("aktivasyon şekli (örnek, kanal):", tuple(S_cal.shape))

# incelenecek kanallar: doğal veride en değişken (en bilgi taşıyan) kanallar
CHANNELS = S_cal.std(0).topk(N_CH).indices.tolist()
ACT_SCALE = {ch: S_cal[:, ch].max().item() for ch in CHANNELS}
print("seçilen kanallar:", CHANNELS)
print("doğal veride kanal başına maksimum skor:", {k: round(v, 2) for k, v in ACT_SCALE.items()})

## 3. Feature visualization (aktivasyon maksimizasyonu)

Ağırlıklar sabit; gürültüden başlayıp birimin skorunu artıran bir görüntü arıyoruz:

$$\mathbf{x}^*=\arg\max_{\mathbf{x}}\; h_z(\mathbf{x}) \;-\; \lambda\,\mathrm{TV}(\mathbf{x})$$

Kitaptaki gibi küçük kaydırma (jitter) ve bir düzenlileştirme (toplam varyasyon, TV) kullanıyorum. Fourier parametrizasyonu ya da GAN önsel bilgisi yok; bu bilinçli bir sadeleştirme.

Bu bölümde **orijinal (temiz) ağ** için FV üretiyorum.

In [ ]:
def tv(x):
    return (x[:, :, 1:, :] - x[:, :, :-1, :]).abs().mean() + (x[:, :, :, 1:] - x[:, :, :, :-1]).abs().mean()

def feature_visualization(probe, channel, act_scale, init_std, steps=FV_STEPS, lr=0.02, seed=0, jitter=8):
    tv_w = 10.0 * act_scale
    probe.channel = channel
    gen = torch.Generator().manual_seed(seed)
    img = (0.5 + init_std * torch.randn(1, 3, 224, 224, generator=gen)).clamp(0, 1).to(DEVICE).requires_grad_(True)
    opt = torch.optim.Adam([img], lr=lr)
    rng = random.Random(seed)
    for _ in range(steps):
        dy, dx = rng.randint(-jitter, jitter), rng.randint(-jitter, jitter)
        x = torch.roll(img, shifts=(dy, dx), dims=(2, 3))
        _, act = probe.forward_all(x)
        loss = -act[:, channel].mean() + tv_w * tv(img)
        opt.zero_grad(); loss.backward(); opt.step()
        with torch.no_grad():
            img.clamp_(0, 1)
    return img.detach()

def hf_energy(x):
    """Girdinin yüksek frekans enerjisi: gri görüntünün Laplace filtresi çıktısının ortalama mutlak değeri."""
    lap = torch.tensor([[0., 1., 0.], [1., -4., 1.], [0., 1., 0.]], device=x.device).view(1, 1, 3, 3)
    return F.conv2d(x.mean(1, keepdim=True), lap).abs().mean(dim=(1, 2, 3))

# FV başlangıç gürültüsü: kapı eşiğinden belirgin büyük HF enerjisi veren en küçük std seçilecek (bölüm 4'te)
e_nat = hf_energy(X_cal)
TAU = HF_MARGIN * e_nat.max().item()
print(f"doğal görüntülerde HF enerjisi: ort={e_nat.mean():.4f}, maks={e_nat.max():.4f} | kapı eşiği TAU={TAU:.4f}")

INIT_STD = 0.3
for s in [0.05, 0.1, 0.15, 0.2, 0.3]:
    xr = (0.5 + s * torch.randn(8, 3, 224, 224)).clamp(0, 1)
    if hf_energy(xr).mean().item() >= 2.5 * TAU:
        INIT_STD = s
        break
print("FV başlangıç gürültü std:", INIT_STD)

FV_CLEAN = {ch: feature_visualization(probe_clean, ch, ACT_SCALE[ch], INIT_STD) for ch in CHANNELS}
print("temiz ağ için FV üretildi.")

## 4. Gizli devre: FV'yi doğal davranışı bozmadan kandırmak (sadeleştirilmiş)

**Fikir (makaleden).** FV optimizasyonu doğal görüntülerin dağılımının *dışında* bir girdi üretir. Ağa, bu dışarıdaki girdiyi tespit edip birimin aktivasyonunu değiştiren bir devre eklenirse:

* doğal görüntülerde devre **hiç tetiklenmez**, ağ aynen eskisi gibi çalışır,
* FV optimizasyonu ise yalnızca devrenin tetiklenmesini maksimize eder ve **anlamsız bir görüntü** üretir.

**Benim sürümüm.** Kapı $g(\mathbf{x}) = \max(0,\; e(\mathbf{x}) - \tau)/\tau$ ve seçili kanal için

$$h'_z(\mathbf{x}) = h_z(\mathbf{x}) + A\cdot g(\mathbf{x}),$$

burada $e(\mathbf{x})$ girdinin yüksek frekans enerjisi, $\tau$ doğal kalibrasyon görüntülerindeki en büyük enerjinin $1.5$ katı ve $A$ doğal veride görülen en yüksek kanal skorunun 3 katı. Doğal görüntülerde $e<\tau$ olduğu sürece $g=0$ olur ve çıktı **birebir aynıdır**.

Bu, makalenin tasarladığı devre değil, aynı mantığın (tetikleyici + aktivasyon ekleme) en basit hâlidir.

In [ ]:
def make_gate(tau):
    def gate(x):
        return F.relu(hf_energy(x) - tau) / tau
    return gate

probes_fool = {}
FV_FOOL = {}
for ch in CHANNELS:
    pf = UnitProbe(copy.deepcopy(base))
    pf.channel, pf.amp, pf.gate = ch, 3.0 * ACT_SCALE[ch], make_gate(TAU)
    probes_fool[ch] = pf
    FV_FOOL[ch] = feature_visualization(pf, ch, ACT_SCALE[ch], INIT_STD)
print("kandırılmış ağlar için FV üretildi.")

## 5. Niceleme: doğal davranış aynı mı, FV neden farklı?

Üç soru:

1. **Doğal davranış değişti mi?** Ayrı test kümesinde kandırılmış ve temiz ağın kanal skorlarını ve logit'lerini karşılaştırıyorum.
2. **FV doğal davranışı yansıtıyor mu?** Temiz birimin kendi FV'sine ve kandırılmış ağın FV'sine verdiği tepkiyi karşılaştırıyorum.
3. **Kandırılmış FV "çok aktive edici" görünüyor mu?** Kandırılmış birimin kendi FV'ye tepkisi doğal en yüksek aktivasyonla kıyaslanıyor.

In [ ]:
S_test_c, L_test_c = unit_scores(probe_clean, X_test)
gate_active = (hf_energy(X_test) > TAU).float().mean().item()
print(f"test görüntülerinin kaçında kapı tetikleniyor: %{100 * gate_active:.1f}")

pred_c = L_test_c.argmax(1)
rows = []
for ch in CHANNELS:
    pf = probes_fool[ch]
    S_test_f, L_test_f = unit_scores(pf, X_test)
    with torch.no_grad():
        pf.channel = ch
        _, a1 = probe_clean.forward_all(FV_CLEAN[ch]);  s_cfv_c = a1[:, ch].mean().item()   # temiz birim, temiz FV
        _, a2 = probe_clean.forward_all(FV_FOOL[ch]);   s_ffv_c = a2[:, ch].mean().item()   # temiz birim, kandırılmış FV
        _, a3 = pf.forward_all(FV_FOOL[ch]);            s_ffv_f = a3[:, ch].mean().item()   # kandırılmış birim, kandırılmış FV
    nat_max = S_cal[:, ch].max().item()
    rows.append(dict(
        kanal=ch,
        dogal_maks=nat_max,
        temiz_birim_temiz_FV=s_cfv_c,
        temiz_birim_kandirilmis_FV=s_ffv_c,
        kandirilmis_birim_kandirilmis_FV=s_ffv_f,
        test_skor_maks_fark=(S_test_c[:, ch] - S_test_f[:, ch]).abs().max().item(),
        test_logit_maks_fark=(L_test_c - L_test_f).abs().max().item(),
        test_tahmin_ayni=(pred_c == L_test_f.argmax(1)).float().mean().item(),
        hf_enerji_temiz_FV=hf_energy(FV_CLEAN[ch]).item(),
        hf_enerji_kandirilmis_FV=hf_energy(FV_FOOL[ch]).item(),
    ))
res = pd.DataFrame(rows).set_index("kanal")
res.T.round(4)

**Nasıl okunur?**

* `test_skor_maks_fark` ve `test_logit_maks_fark` ≈ 0 ve `test_tahmin_ayni` = 1 ise: doğal görüntülerde ağ **değişmedi** (kapı tetiklenmedi).
* `temiz_birim_kandirilmis_FV` küçük, `temiz_birim_temiz_FV` büyükse: kandırılmış FV, birimin gerçek davranışını göstermiyor.
* `kandirilmis_birim_kandirilmis_FV` çok yüksekse: kandırılmış ağın FV'si "bu birimi güçlü aktive eden görüntü" gibi görünüyor, oysa bu görüntü yalnızca gizli devreyi tetikliyor.
* `hf_enerji_*` kapının neden tetiklendiğini gösterir (eşik `TAU`).

## 6. Görsel karşılaştırma: FV vs doğal en yüksek aktive eden görüntüler

Kitaptaki öneri: FV'yi eğitim/doğal örneklerle birlikte göster. Her satırda: **temiz FV**, kalibrasyon kümesinden birimi **en çok aktive eden 3 doğal görüntü** (bu, kandırılmış ağda da aynıdır), ve **kandırılmış ağın FV'si**.

In [ ]:
def timg(x):
    return x[0].detach().cpu().permute(1, 2, 0).numpy().clip(0, 1)

fig, axes = plt.subplots(len(CHANNELS), 5, figsize=(14, 2.9 * len(CHANNELS)))
for r, ch in enumerate(CHANNELS):
    top = S_cal[:, ch].topk(3).indices
    axes[r, 0].imshow(timg(FV_CLEAN[ch]))
    for j in range(3):
        axes[r, 1 + j].imshow(X_cal[top[j]].permute(1, 2, 0).numpy())
    axes[r, 4].imshow(timg(FV_FOOL[ch]))
    axes[r, 0].set_ylabel(f"kanal {ch}")
    if r == 0:
        axes[r, 0].set_title("FV (temiz ağ)")
        for j in range(3):
            axes[r, 1 + j].set_title(f"doğal örnek #{j+1}")
        axes[r, 4].set_title("FV (kandırılmış ağ)")
    for a in axes[r]:
        a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## 7. Sonuç özeti (bu çalıştırmanın gerçek sayılarıyla)

In [ ]:
print(f"""KAPI/DOĞAL DAVRANIŞ
- Test kümesinde kapının tetiklendiği görüntü oranı: %{100*gate_active:.1f}
- Kanal skorunda doğal görüntülerdeki en büyük fark (tüm kanallar): {res.test_skor_maks_fark.max():.6f}
- Logit'lerdeki en büyük fark: {res.test_logit_maks_fark.max():.6f}
- Test tahminlerinin aynı kalma oranı: {res.test_tahmin_ayni.min():.3f}

FV ve GERÇEK DAVRANIŞ (kanal ortalaması)
- Temiz birimin kendi FV'sine skoru: {res.temiz_birim_temiz_FV.mean():.2f}
- Temiz birimin kandırılmış FV'ye skoru: {res.temiz_birim_kandirilmis_FV.mean():.2f}
- Kandırılmış birimin kendi FV'sine skoru: {res.kandirilmis_birim_kandirilmis_FV.mean():.2f}  (doğal maksimum ort.: {res.dogal_maks.mean():.2f})
""")

## 8. Tartışma ve sınırlar

**Ne gösterdik?** Aynı doğal davranışa sahip iki ağdan biri, gizli bir tetikleyici sayesinde feature visualization'ı anlamsız bir görüntüye çevirebiliyor. Yani FV, bir birimin doğal veride ne yaptığının **kanıtı değil**; yalnızca optimizasyonun bulduğu bir girdi. Bu, Molnar'ın kitabındaki "yorumlanabilirlik yanılsaması" uyarısının manipülasyon tarafıdır.

**Bu kopyanın makaleden farkları**
* Makalenin ağ devresi yerine el yapımı, çok basit bir HF-enerji kapısı kullanıldı; makale ayrıca FV'nin *istenen hedef görüntüyü* göstermesini sağlayan daha ince devreler ve teorik analiz içerir (bunları burada yeniden üretmiyorum).
* FV üretimi Lucid/Fourier tabanlı değil; piksel uzayında Adam + jitter + TV. Sonuçlar bu seçimlere duyarlıdır (`FV_STEPS`, `INIT_STD`, TV ağırlığı).
* Tek bir model, tek bir katman ve 4 kanal incelendi; genelleme iddiası yok.
* Bu kapı, doğal görüntüler eşiğin altında kaldığı sürece birebir güvenli; çok yüksek frekanslı doğal bir görüntü eşiği aşarsa aktivasyonu da değişir (test kümesindeki oranı yukarıda raporlandı).

**Savunma için olası sorular**
* *Neden işe yarıyor?* FV doğal dağılımın dışında bir girdi arar; ağ bu bölgede istediğimiz gibi davranacak şekilde değiştirilebilir, doğal bölge ise etkilenmez.
* *Neden doğal örnekler daha güvenilir?* Doğal en yüksek aktive eden örnekler gerçek veri üzerinde ölçülür; ağı değiştiren biri bunları da değiştirmek zorunda kalır (bu da doğal davranışı değiştirir).
* *Nasıl korunulur?* FV'yi tek başına delil saymamak, doğal örneklerle ve nedensel müdahalelerle (kanal ablasyonu, Network Dissection) çapraz doğrulamak, bilinmeyen kaynaklı modellerde dağılım dışı girdilere karşı tutarlılık testi yapmak.
* *Network Dissection ile bağlantı?* ND, birimleri **etiketli doğal görüntülerle** hizalar; bu yüzden bu tür bir FV manipülasyonuna karşı daha az açıktır, ama piksel etiketli veri gerektirir.